## Task 5: Bronze ingestion

In [0]:
from pyspark.sql import functions as F

catalog = "dbr_dev_ua5816bd"
login = "lena066636"

raw_path = f"abfss://{login}@dlsua5816bd.dfs.core.windows.net/raw/"
checkpoint_path = f"abfss://{login}@dlsua5816bd.dfs.core.windows.net/_checkpoints/bronze_orders"
bronze_table = f"{catalog}.{login}_bronze.orders"


In [0]:
sample_csv = """order_id,customer,amount
1,Ivan,25.50
2,Olena,42.00
3,Petro,15.75
"""

dbutils.fs.put(raw_path + "orders_sample.csv", sample_csv, overwrite=True)


In [0]:
df_raw = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", checkpoint_path + "/schema")
    .load(raw_path))


In [0]:
df_bronze = (df_raw
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("load_date", F.current_date()))


In [0]:
(df_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table))


In [0]:
spark.table(bronze_table).show(10, truncate=False)
